In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
import joblib
import warnings
warnings.filterwarnings('ignore')

print("TAHAP 1 FORECASTING: TIME SERIES FEATURE ENGINEERING")

# 1. Load Data
df = pd.read_csv('../data/data_master_clustered.csv')

# 2. Urutkan data berdasarkan Wilayah lalu Tanggal (Wajib untuk Time Series)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by=['region_name', 'date']).reset_index(drop=True)

# 3. Ekstrak Fitur Waktu (Mengubah kalender menjadi angka yang dipahami mesin)
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['dayofyear'] = df['date'].dt.dayofyear
df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)

print("Fitur waktu (Tahun, Bulan, Hari, dll) berhasil diekstrak.")

# 4. Encoding Nama Provinsi (One-Hot Encoding)
# Mesin tidak bisa baca teks "Jawa Timur". Ubah jadi deretan angka biner 0 dan 1.
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_prov = encoder.fit_transform(df[['region_name']])
encoded_cols = encoder.get_feature_names_out(['region_name'])

# Gabungkan hasil encoding ke data utama
df_encoded = pd.DataFrame(encoded_prov, columns=encoded_cols)
df_final = pd.concat([df, df_encoded], axis=1)

# Simpan alat penerjemah provinsi ini ke dapur agar nanti Web bisa memakainya
joblib.dump(encoder, '../models_output/encoder_provinsi.pkl')
print(f"Nama {len(encoded_cols)} Provinsi berhasil diterjemahkan ke angka (One-Hot Encoding).")

# 5. Tentukan Target Regresi (Variabel yang akan diramal untuk kebutuhan EWS)
target_forecast = [
    'Rainfall', 'SPI - 3 months', 'Temperature', 'Water Satisfaction Index (WSI)', 
    'Solar Radiation', 'Soil Moisture (gapfilled historical time series)', 
    'FPAR', 'FPAR - zscore'
]

# 6. Simpan dataset khusus peramalan ini sementara
path_forecast_data = '../data/data_forecast_ready.csv'
df_final.to_csv(path_forecast_data, index=False)

print(f"\n Data siap latih untuk Forecasting berhasil disimpan ke: {path_forecast_data}")
print(f"Total Baris: {df_final.shape[0]}, Total Kolom Baru: {df_final.shape[1]}")

TAHAP 1 FORECASTING: TIME SERIES FEATURE ENGINEERING
Fitur waktu (Tahun, Bulan, Hari, dll) berhasil diekstrak.
Nama 33 Provinsi berhasil diterjemahkan ke angka (One-Hot Encoding).

 Data siap latih untuk Forecasting berhasil disimpan ke: ../data/data_forecast_ready.csv
Total Baris: 28277, Total Kolom Baru: 54


In [2]:
import pandas as pd
import numpy as np
from sklearn.multioutput import MultiOutputRegressor
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib
import warnings
warnings.filterwarnings('ignore')

print("TAHAP 2 FORECASTING: AR-XGBOOST (AUTO-REGRESSIVE LAG FEATURES)")

df = pd.read_csv('../data/data_forecast_ready.csv')
df['date'] = pd.to_datetime(df['date'])

# 1. ATURAN EMAS: Urutkan berdasarkan Wilayah lalu Tanggal
df = df.sort_values(by=['region_name', 'date']).reset_index(drop=True)

# 2. FEATURE ENGINEERING: Menambahkan Lag Features (Fitur Masa Lalu)
# Buat mesin mengingat cuaca 1, 2, dan 3 langkah ke belakang
print("Membuat fitur memori masa lalu (Lag Features)...")
kolom_lag = ['Rainfall', 'Temperature', 'Soil Moisture (gapfilled historical time series)']
for col in kolom_lag:
    for i in [1, 2, 3]:
        # groupby region_name agar lag Jawa Timur tidak mengambil data akhir Jawa Tengah
        df[f'{col}_lag_{i}'] = df.groupby('region_name')[col].shift(i)

# Hapus baris pertama tiap provinsi yang lag-nya kosong (NaN) karena tidak ada data sebelumnya
df = df.dropna().reset_index(drop=True)

# 3. Definisikan X dan y
target_cols = [
    'Rainfall', 'SPI - 3 months', 'Temperature', 'Water Satisfaction Index (WSI)', 
    'Solar Radiation', 'Soil Moisture (gapfilled historical time series)', 
    'FPAR', 'FPAR - zscore'
]

fitur_waktu = ['year', 'month', 'day', 'dayofyear', 'weekofyear']
fitur_provinsi = [col for col in df.columns if col.startswith('region_name_')]
fitur_lag = [col for col in df.columns if '_lag_' in col]

X_cols = fitur_waktu + fitur_provinsi + fitur_lag
X = df[X_cols]
y = df[target_cols]

# 4. Chronological Split (Tidak boleh diacak!)
split_index = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Data Train: {X_train.shape[0]} baris | Data Test: {X_test.shape[0]} baris")

# 5. Training Model Global
print("Melatih Model XGBoost dengan Insting Auto-Regressive... ⏳")
base_xgb = xgb.XGBRegressor(
    n_estimators=300,        # Naikkan sedikit karena fiturnya bertambah
    max_depth=6,
    learning_rate=0.05,      # Turunkan agar belajarnya lebih teliti
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)

multi_model = MultiOutputRegressor(base_xgb)
multi_model.fit(X_train, y_train)

# 6. Evaluasi 
y_pred = multi_model.predict(X_test)
y_pred_df = pd.DataFrame(y_pred, columns=target_cols)

print("\n LAPORAN EVALUASI FORECASTING (STANDAR INTERNASIONAL):")
for i, col in enumerate(target_cols):
    mae = mean_absolute_error(y_test.iloc[:, i], y_pred_df.iloc[:, i])
    rmse = np.sqrt(mean_squared_error(y_test.iloc[:, i], y_pred_df.iloc[:, i]))
    print(f"➤ {col}:")
    print(f"   MAE  = {mae:.3f} | RMSE = {rmse:.3f}")

# Simpan Model Global
path_model = '../models_output/model_forecast_global.pkl'
joblib.dump(multi_model, path_model)
print(f"\n Model Global Auto-Regressive Berhasil Disimpan!")

TAHAP 2 FORECASTING: AR-XGBOOST (AUTO-REGRESSIVE LAG FEATURES)
Membuat fitur memori masa lalu (Lag Features)...
Data Train: 22542 baris | Data Test: 5636 baris
Melatih Model XGBoost dengan Insting Auto-Regressive... ⏳

 LAPORAN EVALUASI FORECASTING (STANDAR INTERNASIONAL):
➤ Rainfall:
   MAE  = 25.569 | RMSE = 32.968
➤ SPI - 3 months:
   MAE  = 0.489 | RMSE = 0.620
➤ Temperature:
   MAE  = 0.267 | RMSE = 0.340
➤ Water Satisfaction Index (WSI):
   MAE  = 1.924 | RMSE = 4.043
➤ Solar Radiation:
   MAE  = 11906.402 | RMSE = 14965.048
➤ Soil Moisture (gapfilled historical time series):
   MAE  = 0.007 | RMSE = 0.009
➤ FPAR:
   MAE  = 3.084 | RMSE = 3.800
➤ FPAR - zscore:
   MAE  = 0.241 | RMSE = 0.301

 Model Global Auto-Regressive Berhasil Disimpan!
